In [20]:
import yfinance as yf
import pandas as pd
import numpy as np

# ==========================================
# STEP 1: DEFINE UNIVERSE AND TIME HORIZON
# ==========================================
# Current Nifty 50 constituents (Ticker format for Yahoo Finance requires '.NS')
nifty50_tickers = [
    "ADANIENT.NS", "ADANIPORTS.NS", "APOLLOHOSP.NS", "ASIANPAINT.NS", "AXISBANK.NS",
    "BAJAJ-AUTO.NS", "BAJFINANCE.NS", "BAJAJFINSV.NS", "BEL.NS", "BPCL.NS",
    "BHARTIARTL.NS", "BRITANNIA.NS", "CIPLA.NS", "COALINDIA.NS", "DRREDDY.NS",
    "EICHERMOT.NS", "GRASIM.NS", "HCLTECH.NS", "HDFCBANK.NS", "HDFCLIFE.NS",
    "HEROMOTOCO.NS", "HINDALCO.NS", "HINDUNILVR.NS", "ICICIBANK.NS", "ITC.NS",
    "INDUSINDBK.NS", "INFY.NS", "JSWSTEEL.NS", "KOTAKBANK.NS", "LT.NS",
    "M&M.NS", "MARUTI.NS", "NTPC.NS", "NESTLEIND.NS", "ONGC.NS",
    "POWERGRID.NS", "RELIANCE.NS", "SBILIFE.NS", "SHRIRAMFIN.NS", "SBIN.NS",
    "SUNPHARMA.NS", "TCS.NS", "TATACONSUM.NS", "TMPV.NS", "TATASTEEL.NS",
    "TECHM.NS", "TITAN.NS", "ULTRACEMCO.NS", "TRENT.NS", "WIPRO.NS"
]

# Benchmark index symbol for Yahoo Finance
BENCHMARK = "^NSEI"  # Nifty 50 Index

# Date range: 2013-01-01 to 2025-12-31 
# Note: Start from 2013 so we have 3 full years of monthly historical data prior to Jan 1, 2016 for computing rolling Beta.
START_DATE = "2013-01-01"
END_DATE = "2025-12-31"

# ==========================================
# STEP 2: DOWNLOAD MONTHLY PRICE DATA
# ==========================================
print("Downloading monthly stock price data...")
stock_prices = yf.download(nifty50_tickers, start=START_DATE, end=END_DATE, interval="1mo")['Close']

print("Downloading benchmark index data...")
benchmark_prices = yf.download(BENCHMARK, start=START_DATE, end=END_DATE, interval="1mo")['Close']

# Calculate monthly percentage returns
stock_returns = stock_prices.pct_change().dropna(how="all")
benchmark_returns = benchmark_prices.pct_change().dropna()

# ==========================================
# STEP 3: EXCLUDE FINANCIAL FIRMS (DATA CLEANING)
# ==========================================
# Banks, NBFCs, and Insurance companies have different balance sheet/earnings structures.
financial_tickers = [
    "AXISBANK.NS", "BAJFINANCE.NS", "BAJAJFINSV.NS", "HDFCBANK.NS", "HDFCLIFE.NS",
    "ICICIBANK.NS", "INDUSINDBK.NS", "KOTAKBANK.NS", "SBILIFE.NS", "SHRIRAMFIN.NS", "SBIN.NS"
]

clean_stock_returns = stock_returns.drop(columns=financial_tickers, errors='ignore')

# Filter stocks with missing data for more than 10% of total months
missing_ratio = clean_stock_returns.isnull().sum() / len(clean_stock_returns)
valid_tickers = missing_ratio[missing_ratio <= 0.10].index

final_stock_returns = clean_stock_returns[valid_tickers]

# ==========================================
# STEP 4: GENERATE DUMMY FUNDAMENTALS FILE
# ==========================================
# Note: Yahoo Finance provides current P/E & Market Cap, but historical annual P/E 
# for Indian stocks over 10 years is best sourced from Screener.in or Capitaline.
# We generate the template below where you can plug in annual Jan 1 attributes.

years = list(range(2016, 2026))
annual_data_list = []

for year in years:
    for ticker in valid_tickers:
        annual_data_list.append({
            "Year": year,
            "Ticker": ticker,
            "Market_Cap_Cr": np.nan,  # Replace with actual historical values (in ₹ Crores)
            "PE_Ratio": np.nan        # Replace with actual historical values
        })

annual_attributes_template = pd.DataFrame(annual_data_list)

# Save cleaned monthly returns & template to CSV
final_stock_returns.to_excel(r"C:\Users\HP\OneDrive - PESUNIVERSITY\Documents\MBA - T3\Investment Management\stock_returns.xlsx")
benchmark_returns.to_excel(r"C:\Users\HP\OneDrive - PESUNIVERSITY\Documents\MBA - T3\Investment Management\benchmark_returns.xlsx")
annual_attributes_template.to_excel(r"C:\Users\HP\OneDrive - PESUNIVERSITY\Documents\MBA - T3\Investment Management\annual_attributes.xlsx", index=False)

print("\n--- PHASE 1 COMPLETE ---")
print(f"Total non-financial stocks retained: {len(valid_tickers)}")
print("Files saved:")
print("1. nifty50_monthly_returns_2013_2025.csv")
print("2. nifty50_benchmark_returns_2013_2025.csv")
print("3. annual_attributes_template.csv (fill Market Cap and P/E here)")

import pandas as pd
df = pd.read_excel(r"C:\Users\HP\OneDrive - PESUNIVERSITY\Documents\MBA - T3\Investment Management\stock_returns.xlsx", index_col=0)
print(df.shape)

[*****                 10%                       ]  5 of 50 completed

[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  1 of 1 completed



--- PHASE 1 COMPLETE ---
Total non-financial stocks retained: 39
Files saved:
1. nifty50_monthly_returns_2013_2025.csv
2. nifty50_benchmark_returns_2013_2025.csv
3. annual_attributes_template.csv (fill Market Cap and P/E here)
(155, 39)


In [22]:
import pandas as pd
import numpy as np

# ==========================================
# 1. FILE PATHS & DATA LOADING
# ==========================================
base_path = r"C:\Users\HP\OneDrive - PESUNIVERSITY\Documents\MBA - T3\Investment Management"

stock_returns = pd.read_excel(f"{base_path}\\stock_returns.xlsx", index_col=0, parse_dates=True)
benchmark_returns = pd.read_excel(f"{base_path}\\benchmark_returns.xlsx", index_col=0, parse_dates=True)
annual_attrs = pd.read_excel(f"{base_path}\\annual_attributes.xlsx")

# Monthly Risk-Free Rate (assuming 6% annual T-bill rate -> 0.5% monthly)
rf_monthly = 0.06 / 12

# Clean tickers to prevent matching issues
annual_attrs['Ticker'] = annual_attrs['Ticker'].astype(str).str.strip()
stock_returns.columns = stock_returns.columns.astype(str).str.strip()

# ==========================================
# 2. CALCULATE 36-MONTH ROLLING STOCK BETAS
# ==========================================
bench_col = benchmark_returns.columns[0]
bench_excess = benchmark_returns[bench_col] - rf_monthly

def calculate_beta(ticker, jan1_date):
    start_date = jan1_date - pd.DateOffset(years=3)
    if ticker not in stock_returns.columns:
        return np.nan
        
    stock_sub = stock_returns.loc[start_date:jan1_date, ticker] - rf_monthly
    bench_sub = bench_excess.loc[start_date:jan1_date]
    
    combined = pd.concat([stock_sub, bench_sub], axis=1).dropna()
    if len(combined) < 24: # Require at least 24 months of data
        return np.nan
        
    cov = np.cov(combined.iloc[:, 0], combined.iloc[:, 1])[0, 1]
    var_bench = np.var(combined.iloc[:, 1])
    
    return cov / var_bench if var_bench != 0 else np.nan

# Compute Betas for every stock on Jan 1 of each year
beta_list = []
years = sorted(annual_attrs['Year'].unique())

for year in years:
    jan1_date = pd.to_datetime(f"{year}-01-01")
    for ticker in stock_returns.columns:
        b_val = calculate_beta(ticker, jan1_date)
        beta_list.append({"Year": year, "Ticker": ticker, "Computed_Beta": b_val})

beta_df = pd.DataFrame(beta_list)

# Merge calculated Betas back into annual_attributes
full_annual = pd.merge(annual_attrs, beta_df, on=["Year", "Ticker"], how="left")

# Clean zero/negative values to avoid NaN/inf issues during log/reciprocal transforms
full_annual['Market_Cap_Cr'] = full_annual['Market_Cap_Cr'].replace(0, np.nan)
full_annual['PE_Ratio'] = full_annual['PE_Ratio'].apply(lambda x: np.nan if pd.isna(x) or x <= 0 else x)

# Derived Attributes: Log Market Cap & Earnings Yield (E/P)
full_annual['Log_Market_Cap'] = np.log(full_annual['Market_Cap_Cr'])
full_annual['Earnings_Yield'] = 1 / full_annual['PE_Ratio']

# ==========================================
# 3. TERCILE SORTING & PORTFOLIO ASSIGNMENT
# ==========================================
def assign_terciles(series):
    valid_series = series.dropna()
    if len(valid_series) < 3:
        return pd.Series(index=series.index, data=np.nan)
    return pd.qcut(series, q=3, labels=["Low", "Medium", "High"], duplicates="drop")

portfolio_assignments = []

for year in years:
    sub_df = full_annual[full_annual['Year'] == year].copy()
    
    # Independent Tercile Sorting
    sub_df['Cap_Bucket'] = assign_terciles(sub_df['Log_Market_Cap'])
    sub_df['Beta_Bucket'] = assign_terciles(sub_df['Computed_Beta'])
    sub_df['PE_Bucket'] = assign_terciles(sub_df['Earnings_Yield'])
    
    portfolio_assignments.append(sub_df)

final_assignments = pd.concat(portfolio_assignments, ignore_index=True)

# Save Phase 2 Outputs
final_assignments.to_excel(f"{base_path}\\portfolio_assignments_2016_2025.xlsx", index=False)
full_annual.to_excel(f"{base_path}\\annual_attributes.xlsx", index=False)

print("--- PHASE 2 COMPLETE ---")
print("Saved files:")
print("1. portfolio_assignments_2016_2025.xlsx")
print("2. annual_attributes.xlsx (updated with Betas & Log values)")

--- PHASE 2 COMPLETE ---
Saved files:
1. portfolio_assignments_2016_2025.xlsx
2. annual_attributes.xlsx (updated with Betas & Log values)


In [23]:
import pandas as pd
import numpy as np

# ==========================================
# 1. FILE PATHS & DATA LOADING
# ==========================================
base_path = r"C:\Users\HP\OneDrive - PESUNIVERSITY\Documents\MBA - T3\Investment Management"

stock_returns = pd.read_excel(f"{base_path}\\stock_returns.xlsx", index_col=0, parse_dates=True)
benchmark_returns = pd.read_excel(f"{base_path}\\benchmark_returns.xlsx", index_col=0, parse_dates=True)
assignments = pd.read_excel(f"{base_path}\\portfolio_assignments_2016_2025.xlsx")

# Monthly Risk-Free Rate (Assuming 6% annual T-bill rate -> 0.5% monthly)
rf_monthly = 0.06 / 12

# Clean tickers to ensure perfect alignment
assignments['Ticker'] = assignments['Ticker'].astype(str).str.strip()
stock_returns.columns = stock_returns.columns.astype(str).str.strip()

# Calculate benchmark excess return
bench_col = benchmark_returns.columns[0]
benchmark_returns['Market_Excess_Return'] = benchmark_returns[bench_col] - rf_monthly

# ==========================================
# 2. MONTHLY PORTFOLIO RETURN AGGREGATION
# ==========================================
portfolio_monthly_list = []
years = sorted(assignments['Year'].unique())

for year in years:
    # Filter portfolio assignments for the rebalancing year
    year_assignments = assignments[assignments['Year'] == year]
    
    # Get monthly stock returns for that year
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"
    monthly_slice = stock_returns.loc[start_date:end_date]
    
    for date, row in monthly_slice.iterrows():
        b_excess = benchmark_returns.loc[date, 'Market_Excess_Return'] if date in benchmark_returns.index else np.nan
        
        row_data = {
            "Date": date,
            "Year": year,
            "Market_Excess_Return": b_excess
        }
        
        # Track portfolio returns for Market Cap, Beta, and P/E Terciles
        bucket_mapping = [('Cap_Bucket', 'Cap'), ('Beta_Bucket', 'Beta'), ('PE_Bucket', 'PE')]
        
        for bucket_col, attr_label in bucket_mapping:
            for bucket_name in ["Low", "Medium", "High"]:
                # Identify tickers belonging to this specific bucket
                bucket_tickers = year_assignments[year_assignments[bucket_col] == bucket_name]['Ticker'].values
                
                # Keep valid tickers with active return data for month t
                valid_tickers = [t for t in bucket_tickers if t in stock_returns.columns and not pd.isna(row[t])]
                
                if len(valid_tickers) > 0:
                    # Equal-weighted mean return across constituent stocks
                    p_return = row[valid_tickers].mean()
                    p_excess = p_return - rf_monthly
                else:
                    p_return = np.nan
                    p_excess = np.nan
                
                row_data[f"{attr_label}_{bucket_name}_Return"] = p_return
                row_data[f"{attr_label}_{bucket_name}_Excess_Return"] = p_excess
                
        portfolio_monthly_list.append(row_data)

# Combine into a single monthly time-series DataFrame
monthly_portfolio_df = pd.DataFrame(portfolio_monthly_list)
monthly_portfolio_df.set_index("Date", inplace=True)

# ==========================================
# 3. SAVE PHASE 3 RESULTS TO EXCEL
# ==========================================
output_file = f"{base_path}\\monthly_portfolio_returns_2016_2025.xlsx"
monthly_portfolio_df.to_excel(output_file)

print("--- PHASE 3 COMPLETE ---")
print(f"Total months tracked: {len(monthly_portfolio_df)}")
print(f"Saved file: monthly_portfolio_returns_2016_2025.xlsx")

--- PHASE 3 COMPLETE ---
Total months tracked: 120
Saved file: monthly_portfolio_returns_2016_2025.xlsx


In [24]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats

# ==========================================
# 1. FILE PATHS & DATA LOADING
# ==========================================
base_path = r"C:\Users\HP\OneDrive - PESUNIVERSITY\Documents\MBA - T3\Investment Management"

monthly_df = pd.read_excel(f"{base_path}\\monthly_portfolio_returns_2016_2025.xlsx", index_col=0, parse_dates=True)
stock_returns = pd.read_excel(f"{base_path}\\stock_returns.xlsx", index_col=0, parse_dates=True)
benchmark_returns = pd.read_excel(f"{base_path}\\benchmark_returns.xlsx", index_col=0, parse_dates=True)
annual_attrs = pd.read_excel(f"{base_path}\\annual_attributes.xlsx")

rf_monthly = 0.06 / 12  # 0.5% monthly risk-free rate
market_excess = monthly_df['Market_Excess_Return']

# Clean string columns
annual_attrs['Ticker'] = annual_attrs['Ticker'].astype(str).str.strip()
stock_returns.columns = stock_returns.columns.astype(str).str.strip()

# ==========================================
# 2. TABLE 2: PORTFOLIO RISK & PERFORMANCE
# ==========================================
table2_rows = []
attributes = ['Cap', 'Beta', 'PE']
terciles = ['Low', 'Medium', 'High']

for attr in attributes:
    for tercile in terciles:
        ret_col = f"{attr}_{tercile}_Return"
        exc_col = f"{attr}_{tercile}_Excess_Return"
        
        if exc_col not in monthly_df.columns:
            continue
            
        p_excess = monthly_df[exc_col].dropna()
        p_return = monthly_df[ret_col].dropna()
        
        # Performance Metrics
        mean_ret_monthly = p_return.mean()
        std_ret_monthly = p_return.std()
        
        ann_return = mean_ret_monthly * 12
        ann_volatility = std_ret_monthly * np.sqrt(12)
        sharpe_ratio = (ann_return - 0.06) / ann_volatility if ann_volatility != 0 else np.nan
        
        # Jensen's Alpha & Portfolio Beta via OLS
        X = sm.add_constant(market_excess.loc[p_excess.index])
        model = sm.OLS(p_excess, X).fit()
        alpha_monthly = model.params['const']
        alpha_annual = alpha_monthly * 12
        beta_p = model.params['Market_Excess_Return']
        alpha_p_value = model.pvalues['const']
        
        table2_rows.append({
            "Attribute": attr,
            "Tercile": tercile,
            "Annualized Return": f"{ann_return * 100:.2f}%",
            "Annualized Volatility": f"{ann_volatility * 100:.2f}%",
            "Sharpe Ratio": round(sharpe_ratio, 3),
            "Jensen's Alpha (Ann.)": f"{alpha_annual * 100:.2f}%",
            "Portfolio Beta": round(beta_p, 3),
            "Alpha p-value": round(alpha_p_value, 4)
        })

table2_df = pd.DataFrame(table2_rows)

# Conduct High vs Low Paired t-tests per attribute
t_test_results = []
for attr in attributes:
    high_exc = monthly_df[f"{attr}_High_Excess_Return"].dropna()
    low_exc = monthly_df[f"{attr}_Low_Excess_Return"].dropna()
    
    t_stat, p_val = stats.ttest_ind(high_exc, low_exc)
    t_test_results.append({
        "Attribute": attr,
        "High vs Low Diff Mean": f"{(high_exc.mean() - low_exc.mean()) * 12 * 100:.2f}%",
        "t-statistic": round(t_stat, 3),
        "p-value": round(p_val, 4)
    })

t_test_df = pd.DataFrame(t_test_results)

# ==========================================
# 3. TABLE 3: FAMA-MACBETH REGRESSIONS
# ==========================================
# Run cross-sectional OLS regressions for each month t
gammas = []
dates = monthly_df.index

for date in dates:
    year = date.year
    # Get yearly fundamental attributes for individual stocks
    year_attrs = annual_attrs[annual_attrs['Year'] == year].copy()
    
    if year_attrs.empty:
        continue
        
    # Get individual stock returns for month t
    if date not in stock_returns.index:
        continue
        
    r_t = stock_returns.loc[date] - rf_monthly
    
    # Merge stock excess return into cross-sectional dataframe
    cs_df = year_attrs.copy()
    cs_df['Excess_Return'] = cs_df['Ticker'].map(r_t)
    
    # Drop rows missing excess returns or explanatory variables
    cs_df = cs_df.dropna(subset=['Excess_Return', 'Log_Market_Cap', 'Computed_Beta', 'Earnings_Yield'])
    
    if len(cs_df) < 10:  # Need adequate cross-sectional observations
        continue
        
    X = cs_df[['Log_Market_Cap', 'Computed_Beta', 'Earnings_Yield']]
    X = sm.add_constant(X)
    y = cs_df['Excess_Return']
    
    model = sm.OLS(y, X).fit()
    gammas.append(model.params)

gammas_df = pd.DataFrame(gammas)

# Calculate Fama-MacBeth Time-Series Averages & t-statistics
fm_summary = []
T = len(gammas_df)

for col in gammas_df.columns:
    mean_gamma = gammas_df[col].mean()
    std_gamma = gammas_df[col].std()
    se_gamma = std_gamma / np.sqrt(T)
    t_stat = mean_gamma / se_gamma
    p_val = stats.t.sf(np.abs(t_stat), df=T - 1) * 2
    
    fm_summary.append({
        "Variable": "Intercept" if col == 'const' else col,
        "Average Coefficient (Gamma)": round(mean_gamma, 5),
        "Standard Error": round(se_gamma, 5),
        "t-statistic": round(t_stat, 3),
        "p-value": round(p_val, 4),
        "Significance": "***" if p_val < 0.01 else ("**" if p_val < 0.05 else ("*" if p_val < 0.1 else "Not Sig"))
    })

table3_df = pd.DataFrame(fm_summary)

# ==========================================
# 4. SAVE RESULTS TO EXCEL
# ==========================================
with pd.ExcelWriter(f"{base_path}\\Phase4_Empirical_Results.xlsx") as writer:
    table2_df.to_excel(writer, sheet_name="Table 2 - Performance Metrics", index=False)
    t_test_df.to_excel(writer, sheet_name="Table 2 - High vs Low Tests", index=False)
    table3_df.to_excel(writer, sheet_name="Table 3 - Fama-MacBeth", index=False)

print("--- PHASE 4 COMPLETE ---")
print("Results successfully exported to 'Phase4_Empirical_Results.xlsx'!")
print("\n--- Summary of Table 3 (Fama-MacBeth Regressions) ---")
print(table3_df.to_string(index=False))

--- PHASE 4 COMPLETE ---
Results successfully exported to 'Phase4_Empirical_Results.xlsx'!

--- Summary of Table 3 (Fama-MacBeth Regressions) ---
      Variable  Average Coefficient (Gamma)  Standard Error  t-statistic  p-value Significance
     Intercept                      0.05239         0.01887        2.776   0.0064          ***
Log_Market_Cap                     -0.00365         0.00155       -2.355   0.0202           **
 Computed_Beta                      0.00648         0.00431        1.503   0.1355      Not Sig
Earnings_Yield                     -0.06129         0.04425       -1.385   0.1687      Not Sig


In [25]:
import pandas as pd
import numpy as np

# ==========================================
# 1. FILE PATHS & DATA LOADING
# ==========================================
base_path = r"C:\Users\HP\OneDrive - PESUNIVERSITY\Documents\MBA - T3\Investment Management"

table2_df = pd.read_excel(f"{base_path}\\Phase4_Empirical_Results.xlsx", sheet_name="Table 2 - Performance Metrics")
ttest_df = pd.read_excel(f"{base_path}\\Phase4_Empirical_Results.xlsx", sheet_name="Table 2 - High vs Low Tests")
table3_df = pd.read_excel(f"{base_path}\\Phase4_Empirical_Results.xlsx", sheet_name="Table 3 - Fama-MacBeth")

# Helper function to parse numeric values safely
def get_sharpe(attr, tercile):
    row = table2_df[(table2_df['Attribute'] == attr) & (table2_df['Tercile'] == tercile)]
    return float(row['Sharpe Ratio'].values[0]) if not row.empty else np.nan

def get_fm_row(var_name):
    row = table3_df[table3_df['Variable'] == var_name]
    return row.iloc[0] if not row.empty else None

# ==========================================
# 2. HYPOTHESIS EVALUATION LOGIC
# ==========================================
results = []

# --- H1: SIZE EFFECT ---
# Small-Cap should outperform Large-Cap (Sharpe_Small > Sharpe_Large) 
# & Log_Market_Cap coefficient (Gamma) should be negative and statistically significant.
sharpe_small = get_sharpe('Cap', 'Low')   # Low Log Market Cap = Small Cap
sharpe_large = get_sharpe('Cap', 'High')  # High Log Market Cap = Large Cap
size_fm = get_fm_row('Log_Market_Cap')

h1_sharpe_cond = sharpe_small > sharpe_large
h1_fm_cond = (size_fm['Average Coefficient (Gamma)'] < 0) and (size_fm['p-value'] < 0.05) if size_fm is not None else False
h1_status = "SUPPORTED" if (h1_sharpe_cond and h1_fm_cond) else ("PARTIALLY SUPPORTED" if (h1_sharpe_cond or h1_fm_cond) else "NOT SUPPORTED")

results.append({
    "Hypothesis": "H1: Size Effect (Small-Cap Anomaly)",
    "Condition 1 (Sharpe Ratio Comparison)": f"Small ({sharpe_small:.3f}) vs Large ({sharpe_large:.3f}) -> {'Pass' if h1_sharpe_cond else 'Fail'}",
    "Condition 2 (Fama-MacBeth Regression)": f"Coeff = {size_fm['Average Coefficient (Gamma)']:.5f}, p = {size_fm['p-value']:.4f} ({size_fm['Significance']})",
    "Empirical Verdict": h1_status,
    "Academic Interpretation": "Small-cap stocks yield a risk-adjusted return premium over large caps if coefficient is negative and significant."
})

# --- H2: LOW-BETA ANOMALY ---
# Low-Beta Sharpe >= High-Beta Sharpe & Beta coefficient is either insignificant or negative (violating standard CAPM).
sharpe_low_beta = get_sharpe('Beta', 'Low')
sharpe_high_beta = get_sharpe('Beta', 'High')
beta_fm = get_fm_row('Computed_Beta')

h2_sharpe_cond = sharpe_low_beta >= sharpe_high_beta
h2_fm_cond = (beta_fm['p-value'] > 0.05) or (beta_fm['Average Coefficient (Gamma)'] <= 0) if beta_fm is not None else False
h2_status = "SUPPORTED" if (h2_sharpe_cond and h2_fm_cond) else ("PARTIALLY SUPPORTED" if (h2_sharpe_cond or h2_fm_cond) else "NOT SUPPORTED")

results.append({
    "Hypothesis": "H2: Low-Beta Anomaly (CAPM Violation)",
    "Condition 1 (Sharpe Ratio Comparison)": f"Low Beta ({sharpe_low_beta:.3f}) vs High Beta ({sharpe_high_beta:.3f}) -> {'Pass' if h2_sharpe_cond else 'Fail'}",
    "Condition 2 (Fama-MacBeth Regression)": f"Coeff = {beta_fm['Average Coefficient (Gamma)']:.5f}, p = {beta_fm['p-value']:.4f} ({beta_fm['Significance']})",
    "Empirical Verdict": h2_status,
    "Academic Interpretation": "Low-beta portfolios deliver equal or superior risk-adjusted returns compared to high-beta portfolios, flattening the Security Market Line (SML)."
})

# --- H3: VALUE EFFECT ---
# High Earnings Yield (Low PE) Sharpe > Low Earnings Yield (High PE) Sharpe
# & Earnings Yield coefficient should be positive and statistically significant.
sharpe_value = get_sharpe('PE', 'High')  # High Earnings Yield = Value (Cheap)
sharpe_growth = get_sharpe('PE', 'Low')  # Low Earnings Yield = Growth (Expensive)
value_fm = get_fm_row('Earnings_Yield')

h3_sharpe_cond = sharpe_value > sharpe_growth
h3_fm_cond = (value_fm['Average Coefficient (Gamma)'] > 0) and (value_fm['p-value'] < 0.05) if value_fm is not None else False
h3_status = "SUPPORTED" if (h3_sharpe_cond and h3_fm_cond) else ("PARTIALLY SUPPORTED" if (h3_sharpe_cond or h3_fm_cond) else "NOT SUPPORTED")

results.append({
    "Hypothesis": "H3: Value Effect (Earnings Yield Premium)",
    "Condition 1 (Sharpe Ratio Comparison)": f"Value/High E/P ({sharpe_value:.3f}) vs Growth/Low E/P ({sharpe_growth:.3f}) -> {'Pass' if h3_sharpe_cond else 'Fail'}",
    "Condition 2 (Fama-MacBeth Regression)": f"Coeff = {value_fm['Average Coefficient (Gamma)']:.5f}, p = {value_fm['p-value']:.4f} ({value_fm['Significance']})",
    "Empirical Verdict": h3_status,
    "Academic Interpretation": "High earnings yield stocks outperform low yield stocks, confirming the value premium in the Nifty 50 sample."
})

hypothesis_df = pd.DataFrame(results)

# ==========================================
# 3. SAVE HYPOTHESIS SUMMARY TO EXCEL
# ==========================================
output_path = f"{base_path}\\Phase5_Hypothesis_Validation_Summary.xlsx"

with pd.ExcelWriter(output_path) as writer:
    hypothesis_df.to_excel(writer, sheet_name="Hypothesis Matrix", index=False)
    table2_df.to_excel(writer, sheet_name="Table 2 Summary", index=False)
    table3_df.to_excel(writer, sheet_name="Table 3 Summary", index=False)

print("--- PHASE 5 COMPLETE ---")
print(f"Results saved to: {output_path}\n")
print("=== EMPIRICAL HYPOTHESIS VALIDATION MATRIX ===")
for idx, row in hypothesis_df.iterrows():
    print(f"\n{row['Hypothesis']}")
    print(f"  • Verdict      : {row['Empirical Verdict']}")
    print(f"  • Sharpe Test  : {row['Condition 1 (Sharpe Ratio Comparison)']}")
    print(f"  • Fama-MacBeth : {row['Condition 2 (Fama-MacBeth Regression)']}")

--- PHASE 5 COMPLETE ---
Results saved to: C:\Users\HP\OneDrive - PESUNIVERSITY\Documents\MBA - T3\Investment Management\Phase5_Hypothesis_Validation_Summary.xlsx

=== EMPIRICAL HYPOTHESIS VALIDATION MATRIX ===

H1: Size Effect (Small-Cap Anomaly)
  • Verdict      : SUPPORTED
  • Sharpe Test  : Small (0.979) vs Large (0.705) -> Pass
  • Fama-MacBeth : Coeff = -0.00365, p = 0.0202 (**)

H2: Low-Beta Anomaly (CAPM Violation)
  • Verdict      : PARTIALLY SUPPORTED
  • Sharpe Test  : Low Beta (0.771) vs High Beta (0.790) -> Fail
  • Fama-MacBeth : Coeff = 0.00648, p = 0.1355 (Not Sig)

H3: Value Effect (Earnings Yield Premium)
  • Verdict      : NOT SUPPORTED
  • Sharpe Test  : Value/High E/P (0.847) vs Growth/Low E/P (1.108) -> Fail
  • Fama-MacBeth : Coeff = -0.06129, p = 0.1687 (Not Sig)
